# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Praveen23-kk/FlyRank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane:** **Ranking Signal Analysis**

**ML Task Type:** **Signal Analysis / Ranking Scoring** (analyzing which safe signals correlate with high visibility and scoring them).

**The One-Paragraph Frame:**
> For **SEO strategists and content creators**, deciding **which content elements to prioritize (e.g., structure, freshness, engagement)**, we will build a **signal analysis and ranked playbook** from **observable search visibility (GSC) and engagement (GA4) data**, analyzing **search visibility (impressions_90d) and clicks** measured by **effect sizes and grouped comparisons (e.g., top vs bottom quartiles)**. A wrong call costs **wasted editorial hours on changes that don't move the needle**. A plain rule isn't enough because **visibility drivers are tangled across multiple metrics like age, word count, and engagement**. We will claim only **observed, directional** results.

**The Decision & The Action:**
- **Decision:** Which content elements should the team audit and improve first?
- **Action:** Generate an evidence-backed report and a prioritized scoring rule based on strong signals.
- **Cost of a Wrong Call:** Wasting editorial writing time on factors that do not correlate with increased visibility.

In [1]:
import os
import pandas as pd
import numpy as np

# Robust path handling (works from repo root or work/notebooks/)
data_path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    data_path = os.path.join("..", "data", "raw", "content_refresh_anonymized.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("..", "..", "data", "raw", "content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print("=" * 60)
print("FLYRANK STARTER DATASET: EXPLORATORY SLICE SUMMARY")
print("=" * 60)
print(f"Total Rows (Content Items) : {len(df):,}")
print(f"Total Columns               : {df.shape[1]}")
print(f"Pseudonymized Clients       : {df['client_id'].nunique()}")
print(f"Unique Content Items (Grain): {df['content_id'].nunique()}")
print("-" * 60)
print("Breakdown by Content Type:")
print(df['content_type'].value_counts(dropna=False).to_string())
print("-" * 60)
print("Breakdown by 90d Trend Direction (Movement Trajectory):")
trend_counts = df['trend_direction'].value_counts(dropna=False)
trend_pcts = df['trend_direction'].value_counts(normalize=True, dropna=False) * 100
summary_trend = pd.DataFrame({'Count': trend_counts, 'Percentage': trend_pcts.round(2)})
print(summary_trend.to_string())

FLYRANK STARTER DATASET: EXPLORATORY SLICE SUMMARY
Total Rows (Content Items) : 30,000
Total Columns               : 44
Pseudonymized Clients       : 32
Unique Content Items (Grain): 30000
------------------------------------------------------------
Breakdown by Content Type:
content_type
keyword article       27207
feedly article         2096
comparison article      697
------------------------------------------------------------
Breakdown by 90d Trend Direction (Movement Trajectory):
                 Count  Percentage
trend_direction                   
down             16262       54.21
stable            5962       19.87
up                4388       14.63
new               2236        7.45
flat              1152        3.84


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target / Proxy Definition:**
In this dataset, our targets for signal correlation are **impressions_90d** (visibility) and **clicks_90d**.

**Observed Outcome vs. Defined Rule:**
- **Observed Outcome:** These targets are empirical search performance measurements from Google Search Console, representing real-world traffic and visibility, not subjective human labels.

**Data Integrity & Leakage Safeguard:**
- We must ensure we don't use direct derivatives of impressions/clicks (like position_tier based exactly on impressions) as predictive features if we are predicting them. We will look at features like content_age_days, word_count, and content_type.

In [2]:
# Analyze the target distribution
high_visibility = df[df['impressions_90d'] > df['impressions_90d'].median()]
print("=" * 60)
print("TARGET ANALYSIS: impressions_90d and clicks_90d")
print("=" * 60)
print(f"Median 90d Impressions: {df['impressions_90d'].median():.0f}")
print(f"Median 90d Clicks: {df['clicks_90d'].median():.0f}")
print(f"Pages above median visibility: {len(high_visibility):,} ({(len(high_visibility)/len(df))*100:.2f}%)")
print("-" * 60)
candidate_features = ['content_age_days', 'days_since_last_update', 'word_count', 'content_type']
print(f"Candidate Safe Feature Subset Sample: {candidate_features}")
print("Leakage Check: Target metrics will not be used to predict themselves.")


TARGET ANALYSIS: impressions_90d and clicks_90d
Median 90d Impressions: 731
Median 90d Clicks: 1
Pages above median visibility: 14,998 (49.99%)
------------------------------------------------------------
Candidate Safe Feature Subset Sample: ['content_age_days', 'days_since_last_update', 'word_count', 'content_type']
Leakage Check: Target metrics will not be used to predict themselves.


## 3. Success metric

*How will you measure if your analysis found a real signal?*

**Success Metric:**
We will measure success by calculating the correlation (e.g., Spearman rank) between features and our visibility metric (impressions_90d). We can also use grouped comparisons (e.g., difference in median impressions for top vs bottom quartiles) to show effect sizes.


In [3]:
import numpy as np
# Simple grouped comparison for an example signal: word_count
word_count_median = df['word_count'].median()
high_wc = df[df['word_count'] > word_count_median]['impressions_90d'].median()
low_wc = df[df['word_count'] <= word_count_median]['impressions_90d'].median()
print("=" * 60)
print("SUCCESS METRIC: GROUPED COMPARISON")
print("=" * 60)
print(f"Median impressions for high word count (> {word_count_median:.0f}): {high_wc:.0f}")
print(f"Median impressions for low word count (<= {word_count_median:.0f}): {low_wc:.0f}")
print(f"Effect size (ratio): {high_wc / low_wc:.2f}x")


SUCCESS METRIC: GROUPED COMPARISON
Median impressions for high word count (> 2877): 1122
Median impressions for low word count (<= 2877): 395
Effect size (ratio): 2.84x


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Definition of the Grain:**
- **One row = One pseudonymized content item (`content_id`) associated with a specific client (`client_id`)**, observed over an aggregated trailing 90-day measurement window.
- **Dataset Scale:** 30,000 unique rows across 32 clients.
- **Identifiers:** `content_id` and `client_id` serve exclusively for grouping, joins, and **client-holdout validation splits** (never as feature inputs).
- **Observable Signal Families per Row:**
  1. *Search Visibility (GSC):* `impressions_90d`, `clicks_90d`, `avg_position`, `ctr`
  2. *User Engagement (GA4):* `sessions_90d`, `pageviews_90d`, `engagement_rate`, `scroll_rate`, `ai_sessions_90d`
  3. *Content & Keyword Metadata:* `content_type`, `main_intent`, `word_count`, `content_age_days`, `days_since_last_update`, `search_volume`, `competition`
  4. *Target:* `is_declining_label`

In [4]:
# Verify grain uniqueness
assert df['content_id'].nunique() == len(df), "Grain violation: content_id is not unique!"

# Select representative columns spanning each signal family
core_columns = [
    'content_id', 'client_id', 'content_type',
    'impressions_90d', 'clicks_90d', 'avg_position', 'ctr',
    'sessions_90d', 'engagement_rate', 'scroll_rate',
    'content_age_days', 'days_since_last_update', 'word_count',
    
]

unit_of_analysis_df = df[core_columns].copy()

print("=" * 60)
print("UNIT OF ANALYSIS: ONE ROW = ONE CONTENT ITEM (90-DAY WINDOW)")
print(f"Shape: {unit_of_analysis_df.shape[0]:,} rows x {unit_of_analysis_df.shape[1]} core columns")
print(f"Unique content_id check: {unit_of_analysis_df['content_id'].nunique():,} unique rows (Passed)")
print("=" * 60)

# Display sample of the real dataframe
display_sample = unit_of_analysis_df.head(6)
print(display_sample.to_string(index=False))

UNIT OF ANALYSIS: ONE ROW = ONE CONTENT ITEM (90-DAY WINDOW)
Shape: 30,000 rows x 13 core columns
Unique content_id check: 30,000 unique rows (Passed)
          content_id         client_id    content_type  impressions_90d  clicks_90d  avg_position  ctr  sessions_90d  engagement_rate  scroll_rate  content_age_days  days_since_last_update  word_count
content_304f48230142 client_f369cb89fc keyword article             3803          29          10.6 0.76            17             5.88         4.55               187                      20      3221.0
content_a1fb4e703a9e client_4e07408562 keyword article            15320           7          20.3 0.05             9             0.00        10.00               445                      25      2481.0
content_9aa793d4d895 client_7f2253d7e2 keyword article            12581          11          36.5 0.09            11             0.00        28.57               141                      20      3515.0
content_331d6c4de07b client_19581e27de keywor

## 5. Why ML beats a fixed rule here

*Why isn't 'just sort by word count' enough?*

**The limitations of a simple rule:**
A simple fixed rule like "longer articles always get more traffic" is naive. Search visibility is determined by a complex interaction between content relevance, competition, age, and engagement metrics. An ML model or a multivariate statistical analysis can uncover these non-linear interactions and isolate the true marginal effect of a signal, controlling for other variables.


In [5]:
# Demonstrate that long articles don't always win
long_but_low = len(df[(df['word_count'] > word_count_median) & (df['impressions_90d'] <= df['impressions_90d'].median())])
short_but_high = len(df[(df['word_count'] <= word_count_median) & (df['impressions_90d'] > df['impressions_90d'].median())])
print("=" * 60)
print("WHY A FIXED RULE FAILS")
print("=" * 60)
print(f"Number of long articles with below-median impressions: {long_but_low:,}")
print(f"Number of short articles with above-median impressions: {short_but_high:,}")
print("This shows that univariate heuristic rules are insufficient to explain search performance.")


WHY A FIXED RULE FAILS
Number of long articles with below-median impressions: 4,927
Number of short articles with above-median impressions: 4,625
This shows that univariate heuristic rules are insufficient to explain search performance.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.